## Checking employment trends in R

In [1]:
# Importing relevant datasets:
# (I installed all of these via Conda, since I was having trouble with
# alternative installation methods on Linux.)
# based on
# https://tidy-survey-r.github.io/tidy-survey-book/c05-descriptive-analysis.html
library(tidyverse)
library(broom)
library(survey)
library(srvyr)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: grid

Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Loading required package: survival


Attaching package: ‘survey’


The following object is masked from ‘package:graphics’:

    dotchart



Attaching package: ‘srvyr’


The following object is masked from ‘package:stats’:

    filter




In [2]:
df <- read_csv('/home/kjb3/D1V1/kjb3largefiles/large_programming/datasets/cps/cps_employment_trends_revised.csv',
col_types = list(year_as_str = col_character()))


Checking the number of rows in the DataFrame:
(See https://dplyr.tidyverse.org/reference/n_distinct.html?q=nrow#null)

In [3]:
nrow(df)

[1] 23488604

In [4]:
# df %>% glimpse()

The following cell provides as somewhat 'hacky' way to specify 2026 as the baseline for an upcoming logistic regression. (It does so by making it the first year when its field is sorted in alphabetical order.)

(See https://dplyr.tidyverse.org/articles/recoding-replacing.html)

In [5]:
df <- df %>% mutate(
year_as_str_last_year_first = year_as_str %>% replace_when(
year_as_str == '2026' ~ '0_2026'))


Creating a survey design object:

(See https://tidy-survey-r.github.io/tidy-survey-book/c10-sample-designs-replicate-weights.html)

In [6]:
sd_df <- df %>% as_survey_design(weights = year_wtfinl, 
strata = final_stratum_revision, ids = cpsid, nest = TRUE)

Creating an object just for 2025 data in order to speed up processing times:

(Since the original CPS data was at the monthly level, I don't think that filtering out years at this stage would distort our results as would, say, filtering the dataset to include *only* software developers.) (See https://tidy-survey-r.github.io/tidy-survey-book/c04-getting-started.html#c04-getting-started for an explanation of why you shouldn't prematurely filter variables out of your sample for a given period.)

In [7]:
df_25 <- df %>% filter(year == 2025)
sd_df_25 <- df_25 %>% as_survey_design(weights = year_wtfinl, 
strata = final_stratum_revision, ids = cpsid, nest = TRUE)
# sd_df_25

In [8]:
nrow(df_25)

[1] 1041831

Creating an object just for 2022 through 2026 (which will prove useful for logistic regressions):

In [9]:
df_22_26 <- df %>% filter(year %in% c(2022, 2023, 2024, 2025, 2026))
sd_df_22_26 <- df_22_26 %>% as_survey_design(weights = year_wtfinl, 
strata = final_stratum_revision, ids = cpsid, nest = TRUE)
# sd_df_22_26

In [10]:
nrow(df_22_26)

[1] 5379704

## Data checks

### 1. Software-developer employment totals by year

In [11]:
sd_df_25_employed_devs_only <- sd_df_25 %>% filter(
occ2010 == 'Software developers, applications and systems software' & employed == 1)

The following code is based on https://tidy-survey-r.github.io/tidy-survey-book/c05-descriptive-analysis.html .

In [12]:
df_dev_emp_2025 = sd_df_25_employed_devs_only %>% survey_count(
year, vartype = c("se", "ci", "var", "cv"))
df_dev_emp_2025

year,n,n_se,n_low,n_upp,n_var,n_cv
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2025,2331220,60146.87,2213263,2449176,3617646432,0.0258006


These standard-error, confidence-itnerval, and CV values match those that I got in svy. (See 'Employment totals by year' section in employment_data_analyses.ipynb.)

### 2. Logistic regression that evaluates whether the proportion of individuals ages 22-65 who are employed as developers has increased significantly from previous years to the current year:

In [13]:
sd_df_22_26_ages_22_65_only <- sd_df_22_26 %>% filter(age >= 22 & age <= 65)

In [14]:
sd_df_ages_22_65_only <- sd_df %>% filter(age >= 22 & age <= 65)

The following logistic-regression code is based on: https://tidy-survey-r.github.io/tidy-survey-book/c07-modeling.html

For a comparison of svy and R's logistic-regression approaches, see: https://svylab.com/learn/notes/posts/svy-vs-r-comparison/index.html

In [15]:
df_dev_emp_prop_by_year_reg_results_full <- sd_df_ages_22_65_only  %>% svyglm(
formula = employed_as_software_developer ~ year_as_str_last_year_first,
design = ., na.action = na.omit, family = quasibinomial)

df_dev_emp_prop_by_year_reg_results_full

Warning message in summary.glm(g):
“observations with zero weight not used for calculating dispersion”
Warning message in summary.glm(glm.object):
“observations with zero weight not used for calculating dispersion”


Stratified 1 - level Cluster Sampling design (with replacement)
With (1400609) clusters.
Called via srvyr
Sampling variables:
  - ids: cpsid 
  - strata: final_stratum_revision 
  - weights: year_wtfinl 

Call:  svyglm(formula = employed_as_software_developer ~ year_as_str_last_year_first, 
    design = ., family = quasibinomial, na.action = na.omit)

Coefficients:
                    (Intercept)  year_as_str_last_year_first2010  
                        -4.3831                          -0.8682  
year_as_str_last_year_first2011  year_as_str_last_year_first2012  
                        -0.8503                          -0.8267  
year_as_str_last_year_first2013  year_as_str_last_year_first2014  
                        -0.8209                          -0.7159  
year_as_str_last_year_first2015  year_as_str_last_year_first2016  
                        -0.6231                          -0.5441  
year_as_str_last_year_first2017  year_as_str_last_year_first2018  
                        -0.50

Not sure why the following tidy() calls are taking so long:

In [16]:
tidy(df_dev_emp_prop_by_year_reg_results_full)

term,estimate,std.error,statistic,p.value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),-4.3831093,0.03082686,-142.184744,0.000000e+00
year_as_str_last_year_first2010,-0.8681984,0.04348948,-19.963413,1.179244e-88
year_as_str_last_year_first2011,-0.8503177,0.04292528,-19.809254,2.546677e-87
year_as_str_last_year_first2012,-0.8266764,0.04265729,-19.379485,1.179417e-83
year_as_str_last_year_first2013,-0.8209008,0.04223490,-19.436552,3.886126e-84
year_as_str_last_year_first2014,-0.7158733,0.04215228,-16.983025,1.113371e-64
year_as_str_last_year_first2015,-0.6231320,0.04166683,-14.955110,1.455484e-50
year_as_str_last_year_first2016,-0.5440654,0.04088611,-13.306851,2.123954e-40
year_as_str_last_year_first2017,-0.5069612,0.04069718,-12.456911,1.288022e-35


The p-values within this output are actually a bit smaller than those within my svy-based Python analysis, even though I *believe* I have the same setup for each function (except that my Python dataset includes more years--which *shouldn't* make a difference here). I might reach out to svy's developer to get his thoughts on what might be going on here. (It's quite possible that the `survey` and `svy` libraries use different default settings for these sorts of regresisons, or that I need to specify the same tolerance setting for both functions; see https://svylab.com/learn/notes/posts/svy-vs-r-comparison/#generalized-linear-models-glms)

## 2. Calculating the proportion of individuals aged 22-65, by year, employed as software developers from 2022 onward

(The following code is based on https://tidy-survey-r.github.io/tidy-survey-book/c05-descriptive-analysis.html#desc-meanprop)

Since `employed_as_software_developer` is a boolean field (with no -1 values), I probably could have used `survey_mean()` here instead.

In [18]:
df_dev_employment_proportion_by_year <- sd_df_22_26_ages_22_65_only %>% group_by(
year_as_str, employed_as_software_developer) %>% summarize(
p = survey_prop(vartype = c("se", "ci", "var", "cv")))
df_dev_employment_proportion_by_year

When `proportion` is unspecified, `survey_prop()` now defaults to `proportion = TRUE`.
ℹ This should improve confidence interval coverage.
This message is displayed once per session.
Warning message:
“There were 20 warnings in `dplyr::summarise()`.
The first warning was:
ℹ In argument: `p = survey_prop(vartype = c("se", "ci", "var", "cv"))`.
ℹ In group 1: `year_as_str = "2022"`, `employed_as_software_developer = 0`.
Caused by warning in `summary.glm()`:
! observations with zero weight not used for calculating dispersion
ℹ Run `dplyr::last_dplyr_warnings()` to see the 19 remaining warnings.”


year_as_str,employed_as_software_developer,p,p_se,p_low,p_upp,p_var,p_cv
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2022,0,0.98950314,0.0002675990,0.988965479,0.99001487,7.160921e-08,0.0002704377
2022,1,0.01049686,0.0002675990,0.009985131,0.01103452,7.160921e-08,0.0254932421
2023,0,0.98942517,0.0002699050,0.988882857,0.98994129,7.284872e-08,0.0002727897
2023,1,0.01057483,0.0002699050,0.010058709,0.01111714,7.284872e-08,0.0255233354
2024,0,0.98960437,0.0002679554,0.989065845,0.99011664,7.180009e-08,0.0002707702
2024,1,0.01039563,0.0002679554,0.009883358,0.01093416,7.180009e-08,0.0257757776
2025,0,0.98889055,0.0002916135,0.988304220,0.98944781,8.503843e-08,0.0002948896
2025,1,0.01110945,0.0002916135,0.010552191,0.01169578,8.503843e-08,0.0262491499
2026,0,0.98766752,0.0003749258,0.986910634,0.98838115,1.405694e-07,0.0003796073


## 2. Calculating the proportion of individuals, by age range, employed as software developers from 2022 onward

(Since we'd like to evaluate proportions for all age ranges, we'll remove the age filter applied in the previous analysis.)

In [ ]:
df_dev_employment_proportion_by_year_and_age_range <- sd_df_22_26 %>% group_by(
year_as_str, age_range, employed_as_software_developer) %>% summarize(
p = survey_prop(vartype = c("se", "ci", "var", "cv")))
df_dev_employment_proportion_by_year_and_age_range

### 3. Logistic regression that evaluates whether software employment *by age group* has increased significantly from previous years to the most recent year

(One option would be to loop through a list of all age ranges, then perform a logistic regression for each. However, to speed things up, I'll simply perform standalone regressions for at least some of the age ranges for which I found, in my Python-based analysis, a significant difference in employment proportions between 2022 and 2026. (I'll focus on age ranges with relatively high, though still-significant, p-values in order to check whether my significance findings in Python were correct.)


In [19]:
df_dev_emp_prop_by_year_reg_results_35_39 <- sd_df_22_26  %>% filter (
age_range == '35-39') %>% svyglm(
formula = employed_as_software_developer ~ year_as_str_last_year_first,
design = ., na.action = na.omit, family = quasibinomial)

df_dev_emp_prop_by_year_reg_results_35_39

Warning message in summary.glm(g):
“observations with zero weight not used for calculating dispersion”
Warning message in summary.glm(glm.object):
“observations with zero weight not used for calculating dispersion”


Stratified 1 - level Cluster Sampling design (with replacement)
With (71820) clusters.
Called via srvyr
Sampling variables:
  - ids: cpsid 
  - strata: final_stratum_revision 
  - weights: year_wtfinl 

Call:  svyglm(formula = employed_as_software_developer ~ year_as_str_last_year_first, 
    design = ., family = quasibinomial, na.action = na.omit)

Coefficients:
                    (Intercept)  year_as_str_last_year_first2022  
                        -3.9891                          -0.1873  
year_as_str_last_year_first2023  year_as_str_last_year_first2024  
                        -0.1317                          -0.2715  
year_as_str_last_year_first2025  
                        -0.1286  

Degrees of Freedom: 347099 Total (i.e. Null);  71167 Residual
Null Deviance:	    57070 
Residual Deviance: 57020 	AIC: NA

In [21]:
tidy(df_dev_emp_prop_by_year_reg_results_35_39)

term,estimate,std.error,statistic,p.value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),-3.9891323,0.07082872,-56.320826,0.000000000
year_as_str_last_year_first2022,-0.1872802,0.09342984,-2.004501,0.045020165
year_as_str_last_year_first2023,-0.1317164,0.09400853,-1.401112,0.161185093
year_as_str_last_year_first2024,-0.2715364,0.09497957,-2.858893,0.004252456
year_as_str_last_year_first2025,-0.1286275,0.08199279,-1.568766,0.116706977


In [20]:
df_dev_emp_prop_by_year_reg_results_55_59 <- sd_df_22_26  %>% filter (
age_range == '55-59') %>% svyglm(
formula = employed_as_software_developer ~ year_as_str_last_year_first,
design = ., na.action = na.omit, family = quasibinomial)

df_dev_emp_prop_by_year_reg_results_55_59

Warning message in summary.glm(g):
“observations with zero weight not used for calculating dispersion”
Warning message in summary.glm(glm.object):
“observations with zero weight not used for calculating dispersion”


Stratified 1 - level Cluster Sampling design (with replacement)
With (67528) clusters.
Called via srvyr
Sampling variables:
  - ids: cpsid 
  - strata: final_stratum_revision 
  - weights: year_wtfinl 

Call:  svyglm(formula = employed_as_software_developer ~ year_as_str_last_year_first, 
    design = ., family = quasibinomial, na.action = na.omit)

Coefficients:
                    (Intercept)  year_as_str_last_year_first2022  
                        -4.7762                          -0.2891  
year_as_str_last_year_first2023  year_as_str_last_year_first2024  
                        -0.3067                          -0.2018  
year_as_str_last_year_first2025  
                        -0.2329  

Degrees of Freedom: 335052 Total (i.e. Null);  67202 Residual
Null Deviance:	    27450 
Residual Deviance: 27420 	AIC: NA

In [22]:
tidy(df_dev_emp_prop_by_year_reg_results_55_59)

term,estimate,std.error,statistic,p.value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),-4.7762446,0.1023565,-46.662826,0.00000000
year_as_str_last_year_first2022,-0.2890900,0.1387813,-2.083061,0.03724941
year_as_str_last_year_first2023,-0.3066752,0.1397833,-2.193933,0.02824364
year_as_str_last_year_first2024,-0.2018373,0.1357339,-1.487007,0.13701753
year_as_str_last_year_first2025,-0.2329222,0.1192608,-1.953050,0.05081780
